[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/xla/lab-x3-collectives-in-the-lowering.ipynb)

# LAB·X3 · Collectives in the lowering

**Hardware:** any machine. Eight CPU devices stand in for eight chips; nothing here needs an accelerator.

You never wrote a collective by hand in the JAX path. You wrote a `Mesh` and a `PartitionSpec`, and the partitioner inserted whatever your program's math required. This lab takes one matmul, shards it three different ways, and reads the collective XLA chose for each one straight out of the compiled text: an all-gather, an all-reduce, and a reduce-scatter, one per sharding.

Before you run anything: for each of the three shardings below, guess which of those three collectives it needs, or whether it needs none at all.

1. Shard the first matmul operand along its rows; leave the second operand whole; ask for a fully replicated answer.
2. Shard the contracting dimension on both operands; ask for a fully replicated answer.
3. Shard the contracting dimension the same way as (2), but ask for an answer that is itself sharded along the same axis as (1).

Write your three guesses down before the next cell runs.

In [ ]:
import os
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"

import jax
import jax.numpy as jnp
from jax.sharding import Mesh, PartitionSpec as P, NamedSharding
from jax.experimental.shard_map import shard_map

print(jax.devices())
assert len(jax.devices()) == 8, "expected 8 host-platform devices; check XLA_FLAGS was set before import jax"

mesh = Mesh(jax.devices(), ("d",))
M, K, N = 512, 512, 512

def matmul(a, b):
    return a @ b

a = jnp.ones((M, K), dtype=jnp.float32)
b = jnp.ones((K, N), dtype=jnp.float32)

def compiled_text(in_specs, out_spec):
    in_sh = tuple(NamedSharding(mesh, s) for s in in_specs)
    out_sh = NamedSharding(mesh, out_spec)
    fn = jax.jit(matmul, in_shardings=in_sh, out_shardings=out_sh)
    return fn.lower(a, b).compile().as_text()

def find_collectives(text):
    return [op for op in ("all-reduce", "all-gather", "reduce-scatter") if op in text]

**your prediction:**

In [ ]:
# sharding 1: A's rows split across devices, B kept whole, answer replicated
text_1 = compiled_text([P("d", None), P(None, None)], P(None, None))
print("sharding 1 (shard A's rows, replicate B, replicate output):", find_collectives(text_1))

# sharding 2: the contracting dimension split on both operands, answer replicated
text_2 = compiled_text([P(None, "d"), P("d", None)], P(None, None))
print("sharding 2 (shard the contracting dim on both, replicate output):", find_collectives(text_2))

Take sharding 1 first. Each device already holds a complete, non-overlapping slice of the answer's rows the moment it multiplies its own slice of A by the whole of B. No device's number needs correcting by anyone else's; there is nothing to sum. But every device asked for the full, replicated answer, and no device holds the rows another device computed. All-gather is exactly that operation: collect each device's shard and hand every participant the concatenation.

Sharding 2 is a different problem. Splitting the contracting dimension means every device computes only a partial dot product over its own slice of K, and a partial dot product is not a valid answer on its own: every device's partial has to be added to every other device's partial before the result means anything. All-reduce sums across devices and delivers the same, complete total back to all of them. Nobody's number was final until everyone's number was added in.

**your prediction:**
Sharding 3 keeps sharding 2's inputs. What should change if the requested output is sharded instead of replicated?

In [ ]:
# sharding 3: same input sharding as (2), but the answer is requested sharded
# on the same axis as (1) rather than replicated. Auto-partitioning does not
# reliably fuse this into a single reduce-scatter on every backend; writing
# the collective explicitly with shard_map gets the fused op every time.
def local_matmul(a_shard, b_shard):
    partial = a_shard @ b_shard
    return jax.lax.psum_scatter(partial, "d", scatter_dimension=0, tiled=True)

fn_3 = jax.jit(shard_map(local_matmul, mesh=mesh,
                         in_specs=(P(None, "d"), P("d", None)),
                         out_specs=P("d", None)))
text_3 = fn_3.lower(a, b).compile().as_text()
print("sharding 3 (same inputs as 2, output sharded, written with shard_map + psum_scatter):", find_collectives(text_3))

Sharding 3 asks for the same partial sums as sharding 2, but each device only wants its own slice of the final rows, not the whole array. Doing a full all-reduce and then throwing away the rows you did not ask for wastes exactly the bandwidth spent shipping those discarded rows to you in the first place. Reduce-scatter does the sum and the split as one collective: every device ends up with only the rows it asked for, and the bytes for everyone else's rows never cross the wire to it at all.

Three shardings of the identical matmul, three different collectives, and each one traces straight back to what the requested output actually needed: everything (all-gather), the sum of everything (all-reduce), or just your own slice of the sum (reduce-scatter).

In [ ]:
for line in text_2.splitlines():
    if "all-reduce(" in line:
        print(line.strip()[:200])
        break

`replica_groups=[1,8]<=[8]` and `channel_id=1` are the literal contract sharding 2's all-reduce runs on: every device in that one group of eight must issue this exact instruction, tagged with this exact channel, before any of them can move on. That is what a collective is: not a function call one device makes, an agreement every participant has to honor identically.

**your prediction:**
In a real distributed job, one device issues its all-reduce a beat late, or issues a different collective at that point in its program. What happens to the other seven?

They wait. A collective only completes once every participant in its replica group has issued the matching instruction; a device that never gets there, or gets there having issued something else, leaves the rest of the group blocked on a handshake that is never going to arrive. Nothing crashes. Nothing times out on its own. The fleet just stops, each device parked on a wait it cannot resolve by itself. This is the same failure two layers below matmul sharding, at the level of raw semaphores in a hand-written kernel: a send waiting on a recv that is itself waiting on that same send. GSPMD's job is to make sure every device in a replica group always gets the matching instruction in the matching order; when that invariant holds, you get exactly the collectives you saw above, and when it does not, this is the failure mode you are choosing between.

## mark it run

Chapter 07 (kernels.rudrite.com/xla/spmd) is the partitioner choosing all-gather and all-reduce from nothing but a `Mesh` and a `PartitionSpec`, the mechanism behind the surface the JAX path already taught you. Chapter 08 (kernels.rudrite.com/xla/collectives) is the contract you just wrote down: what every participant in a replica group owes every other participant, and what happens the moment one of them does not pay it.